# Scenario 6 — Adversarial / Obfuscated Phishing Robustness
## Encoder Notebook (BERT Family) — FIXED

**Hypothesis:** Decoder LLMs (semantic reasoning) are more robust to adversarial perturbations than Encoder models (token pattern matching).

**Models compared (all fine-tuned independently):**
- `bert-base-uncased` — original BERT
- `roberta-base` — robustly optimised BERT (more training data, no NSP)
- `distilbert-base-uncased` — 40% smaller distilled BERT, ~60% faster

**Each model is:**
1. Fine-tuned independently on the same clean training data (3 epochs)
2. Evaluated on the same clean test set
3. Evaluated on the same adversarial test set
4. Results shown side by side — best F1 drop used for final hypothesis test

**Perturbations:** Synonym substitution · Character noise · Whitespace injection · Homoglyphs

**NOTE:** Run this notebook first — it saves X_te, X_adv, y_te and encoder results to disk
so the decoder notebook uses the exact same test/adversarial data.

In [ ]:
!nvidia-smi
!pip install -q "numpy==1.26.4" "scipy==1.12.0"
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes peft datasets scikit-learn pandas tqdm

In [1]:
import os, warnings, random, string
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           get_linear_schedule_with_warmup)
from torch.optim import AdamW
from datasets import load_dataset
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── BERT family models ────────────────────────────────────────────────────────
ENCODER_MODELS = {
    "BERT":       "bert-base-uncased",
    "RoBERTa":    "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
}

# ── Shared evaluation function ────────────────────────────────────────────────
def evaluate(y_true, y_pred, name=""):
    return {
        "Model":     name,
        "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1":        f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}"
    }

# ── Dataset class ─────────────────────────────────────────────────────────────
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.enc = tokenizer(
            list(texts), padding="max_length", truncation=True,
            max_length=max_len, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

Device: cuda
GPU: Tesla T4


In [2]:
# ── Load dataset ──────────────────────────────────────────────────────────────
print("Loading ealvaradob/phishing-dataset...")
fs = HfFileSystem()
all_files = fs.glob("datasets/ealvaradob/phishing-dataset/**")
texts_files = [f for f in all_files if "text" in f.lower() and f.endswith(".json")]
hf_url = "hf://" + texts_files[0]
ds = load_dataset("json", data_files={"train": hf_url}, split="train")
df = ds.to_pandas()
df["text"] = df["text"].astype(str).str.strip()
print(f"Total: {len(df):,} | Benign: {(df.label==0).sum():,} | Phishing: {(df.label==1).sum():,}")

X, y = df["text"].values, df["label"].values
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_te, y_val, y_te  = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print(f"Train {len(X_tr):,} | Val {len(X_val):,} | Test {len(X_te):,}")

Loading ealvaradob/phishing-dataset...


texts.json:   0%|          | 0.00/52.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total: 20,137 | Benign: 12,465 | Phishing: 7,672
Train 14,095 | Val 3,021 | Test 3,021


In [3]:
# ── Perturbation function ─────────────────────────────────────────────────────
# Seed is fixed globally above — X_adv is fully deterministic.
# Saved to disk so the decoder notebook evaluates on byte-for-byte
# identical adversarial samples.

SYNONYMS = {
    "verify": "confirm",   "account": "profile",    "click": "select",
    "urgent": "important", "password": "credential", "login": "sign-in",
    "suspend": "restrict", "update": "refresh",      "confirm": "validate",
    "immediately": "promptly", "bank": "financial institution"
}
HOMOGLYPHS = {'a':'а','e':'е','o':'о','p':'р','c':'с','i':'і','x':'х'}

def perturb(text):
    # 1. Synonym substitution
    words = text.split()
    for i, w in enumerate(words):
        if w.lower() in SYNONYMS and random.random() < 0.15:
            words[i] = SYNONYMS[w.lower()]
    text = " ".join(words)
    # 2. Character-level noise
    chars = list(text)
    for i in range(len(chars)):
        if chars[i].isalpha() and random.random() < 0.03:
            chars[i] = random.choice(string.ascii_lowercase)
    text = "".join(chars)
    # 3. Whitespace injection
    words = text.split(); result = []
    for w in words:
        if len(w) > 4 and random.random() < 0.05:
            result.append(w[:len(w)//2] + " " + w[len(w)//2:])
        else:
            result.append(w)
    text = " ".join(result)
    # 4. Homoglyph substitution
    return "".join(HOMOGLYPHS.get(c, c) if random.random() < 0.08 else c for c in text)

print("Generating adversarial test set (full test split)...")
X_adv = np.array([perturb(t) for t in X_te])
print("Original   :", X_te[0][:100])
print("Adversarial:", X_adv[0][:100])

# Save — decoder notebook loads these exact arrays
np.save("/kaggle/working/X_te.npy",  X_te)
np.save("/kaggle/working/X_adv.npy", X_adv)
np.save("/kaggle/working/y_te.npy",  y_te)
print("\nSaved X_te, X_adv, y_te → /kaggle/working/")

Generating adversarial test set (full test split)...
Original   : donnellan complete citation a summary of my request for the complete citation : keith s . donnellan 
Adversarial: dinnellan complеtе citation a summary of my request for the cоmplete citation : keith s . donnеllan 

Saved X_te, X_adv, y_te → /kaggle/working/


In [5]:
# ── Fine-tune and evaluate each BERT family model ────────────────────────────

def train_and_eval(model_id, short_name, epochs=3, batch_size=32, lr=2e-5, max_len=256):
    """Fine-tune one BERT-family model and return clean/adversarial results."""
    print(f"\n{'='*60}")
    print(f"Training: {short_name}  ({model_id})")
    print(f"{'='*60}")

    tok   = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(
                model_id, num_labels=2).to(DEVICE)

    tr_dl = DataLoader(
        EmailDataset(X_tr, y_tr, tok, max_len=max_len),
        batch_size=batch_size, shuffle=True
    )
    opt   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, len(tr_dl) // 5, len(tr_dl) * epochs)

    for ep in range(epochs):
        model.train(); total_loss = 0
        for batch, lbl in tqdm(tr_dl, desc=f"{short_name} ep{ep+1}/{epochs}"):
            loss = model(
                **{k: v.to(DEVICE) for k, v in batch.items()},
                labels=lbl.to(DEVICE)
            ).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total_loss += loss.item()
        print(f"  Epoch {ep+1} avg loss: {total_loss / len(tr_dl):.4f}")

    # Inference helper
    def predict(texts):
        model.eval(); preds = []
        for i in range(0, len(texts), batch_size):
            enc = tok(
                list(texts[i:i+batch_size]), padding=True,
                truncation=True, max_length=max_len, return_tensors="pt"
            ).to(DEVICE)
            with torch.no_grad():
                preds.extend(model(**enc).logits.argmax(-1).cpu().numpy())
        return preds

    p_clean = predict(X_te)
    p_adv   = predict(X_adv)
    del model; torch.cuda.empty_cache()

    r_clean = evaluate(y_te, p_clean, f"{short_name} (clean)")
    r_adv   = evaluate(y_te, p_adv,   f"{short_name} (adversarial)")
    drop    = float(r_clean["F1"]) - float(r_adv["F1"])
    print(f"  → Clean F1: {r_clean['F1']}  Adv F1: {r_adv['F1']}  Drop: -{drop:.4f}")
    return r_clean, r_adv, drop


# ── Run all three ─────────────────────────────────────────────────────────────
all_rows    = []   # for summary table
model_drops = {}   # {name: {clean_f1, adv_f1, drop}}

for short_name, model_id in ENCODER_MODELS.items():
    r_clean, r_adv, drop = train_and_eval(model_id, short_name)
    all_rows.append({**r_clean, "F1 drop": "—"})
    all_rows.append({**r_adv,   "F1 drop": f"-{drop:.4f}"})
    model_drops[short_name] = {
        "model_id":  model_id,
        "clean_f1":  float(r_clean["F1"]),
        "adv_f1":    float(r_adv["F1"]),
        "drop":      drop
    }


Training: BERT  (bert-base-uncased)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT ep1/3:   0%|          | 0/441 [00:00<?, ?it/s]

  Epoch 1 avg loss: 0.1925


...

  Epoch 1 avg loss: 0.1815


DistilBERT ep2/3:   0%|          | 0/441 [00:00<?, ?it/s]

  Epoch 2 avg loss: 0.0425


DistilBERT ep3/3:   0%|          | 0/441 [00:00<?, ?it/s]

  Epoch 3 avg loss: 0.0161
  → Clean F1: 0.9715  Adv F1: 0.9467  Drop: -0.0248


In [6]:
# ── Side-by-side summary table ────────────────────────────────────────────────
df_summary = pd.DataFrame(all_rows)

print("\n" + "="*72)
print("SCENARIO 6 — BERT FAMILY RESULTS (side by side)")
print("All models fine-tuned on same data | Same test & adversarial splits")
print("="*72)
print(df_summary[["Model", "Accuracy", "Precision", "Recall", "F1", "F1 drop"]].to_string(index=False))

# ── Robustness ranking ────────────────────────────────────────────────────────
print("\n" + "-"*72)
print("ROBUSTNESS RANKING  (smaller F1 drop = more robust under perturbation)")
print("-"*72)
ranked = sorted(model_drops.items(), key=lambda x: x[1]["drop"])
for rank, (name, res) in enumerate(ranked, 1):
    print(f"  {rank}. {name:12s}  Clean F1: {res['clean_f1']:.4f}  "
          f"Adv F1: {res['adv_f1']:.4f}  Drop: -{res['drop']:.4f}")

best_name, best_res = ranked[0]
print(f"\n→ Most robust encoder: {best_name}  (F1 drop: -{best_res['drop']:.4f})")
print(  "  This result will be used in the decoder notebook for the hypothesis test.")

# ── Save results for decoder notebook ────────────────────────────────────────
np.save("/kaggle/working/encoder_all_results.npy",  model_drops)
np.save("/kaggle/working/encoder_best_result.npy",  {"name": best_name, **best_res})
print("\nSaved encoder results → /kaggle/working/")


SCENARIO 6 — BERT FAMILY RESULTS (side by side)
All models fine-tuned on same data | Same test & adversarial splits
                   Model Accuracy Precision Recall     F1 F1 drop
            BERT (clean)   0.9851    0.9851 0.9757 0.9804       —
      BERT (adversarial)   0.9639    0.9292 0.9800 0.9539 -0.0265
         RoBERTa (clean)   0.9858    0.9834 0.9791 0.9813       —
   RoBERTa (adversarial)   0.9772    0.9632 0.9774 0.9702 -0.0111
      DistilBERT (clean)   0.9782    0.9657 0.9774 0.9715       —
DistilBERT (adversarial)   0.9580    0.9156 0.9800 0.9467 -0.0248

------------------------------------------------------------------------
ROBUSTNESS RANKING  (smaller F1 drop = more robust under perturbation)
------------------------------------------------------------------------
  1. RoBERTa       Clean F1: 0.9813  Adv F1: 0.9702  Drop: -0.0111
  2. DistilBERT    Clean F1: 0.9715  Adv F1: 0.9467  Drop: -0.0248
  3. BERT          Clean F1: 0.9804  Adv F1: 0.9539  Drop: -0.0265

→